# Lesson 3.2 · Cross-validation and leakage

Cross-validation estimates how well a model generalizes, but **only if the scheme for splitting the data mirrors the real use of the
model**. If the rows of a validation fold look too much like those of training (same person, same period) the
estimate is inflated, even if every step looks correct.

**What you will learn**

- K-fold, stratified and repeated, and what `cross_val_score`, `cross_validate` and `cross_val_predict` measure.
- Why `GroupKFold` and `TimeSeriesSplit` are needed, and how much it costs to use the wrong splitter.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import (GroupKFold, KFold, RepeatedStratifiedKFold, StratifiedKFold, TimeSeriesSplit,
                                     cross_val_predict, cross_val_score, cross_validate)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

d = ds.loans(with_missing=False)
X = d[["age", "income", "job_tenure", "amount"]]
y = d["defaulted"]

## K-fold and stratification

With imbalanced classes (here defaults are about 19%) a plain K-fold can produce folds with different proportions.
`StratifiedKFold` keeps the proportion of the classes in every fold. For a classification it is the default of `cross_val_score`
when you pass an integer.

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
for name, cv in [("KFold(5)", KFold(5, shuffle=True, random_state=0)),
                 ("StratifiedKFold(5)", StratifiedKFold(5, shuffle=True, random_state=0)),
                 ("repeated 5x3", RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0))]:
    s = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")
    print(f"{name:20s} AUC {s.mean():.3f} +/- {s.std():.3f}  ({len(s)} evaluations)")

## More metrics, and the out-of-fold predictions

`cross_validate` computes several metrics and can also return the scores on the training set (`return_train_score=True`): the gap between
training and validation is the signal of overfitting. `cross_val_predict` returns for every row the prediction made by the
model that **did not** see it (out-of-fold predictions), useful for analyzing errors and for stacking.

In [ ]:
r = cross_validate(model, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0),
                   scoring={"auc": "roc_auc", "precision": "precision"}, return_train_score=True)
print({k: round(v.mean(), 3) for k, v in r.items() if k.startswith(("test", "train"))})
oof = cross_val_predict(model, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")[:, 1]
print("out-of-fold predictions:", oof.shape, "| overall AUC:", round(roc_auc_score(y, oof), 3))

## When the random splitter lies: repeated measurements

Sixty patients, ten measurements each. The measurements of the same patient are almost identical to each other (they are their
"fingerprint"), and the outcome depends mostly on an individual characteristic **that no measurement explains**. A model that recognizes
the patient from the fingerprint remembers their outcome.

In [ ]:
p = ds.patient_measurements()
Xp, yp = p[["m1", "m2", "m3"]], p["outcome"]
rf = RandomForestRegressor(100, random_state=0)
random_cv = cross_val_score(rf, Xp, yp, cv=KFold(5, shuffle=True, random_state=0), scoring="r2")
by_group = cross_val_score(rf, Xp, yp, cv=GroupKFold(5), groups=p["patient"], scoring="r2")
print(f"random KFold : R2 {random_cv.mean():.2f}   <- the same patient is in training and in validation")
print(f"GroupKFold   : R2 {by_group.mean():.2f}   <- new patients in validation: the real use situation")

With a new patient the model has nothing to remember and the R² is negative. The correct estimate is the second: if in production
never-seen patients will arrive, the validation must use never-seen patients. `GroupKFold` guarantees that all the rows of a
group stay in the same fold.

## When the random splitter lies: time

In [ ]:
s = ds.time_series()
Xs, ys = s[["t", "s", "c"]], s["y"]
random_cv = cross_val_score(rf, Xs, ys, cv=KFold(5, shuffle=True, random_state=0), scoring="r2")
temporal = cross_val_score(rf, Xs, ys, cv=TimeSeriesSplit(5), scoring="r2")
print(f"random KFold    : R2 {random_cv.mean():.2f}   <- interpolates between the past and the immediate future")
print(f"TimeSeriesSplit : R2 {temporal.mean():.2f}   <- trains on the past, validates on the future")

With a random KFold every validation month has its neighbors in the training set: the model interpolates. In production it will have to predict
the future without having seen anything of what comes later. `TimeSeriesSplit` always trains on the past and validates on the future.

> **Rule of thumb.** Ask yourself "how will this model be used?" and build the splitter so that the validation imitates it:
> new customers → groups by customer; future → temporal order; rare classes → stratification.

## Check your understanding

> *Click on a question to show its answer.*

<details><summary>Why is `return_train_score=True` useful?</summary>

Because the gap between the training score and the validation score measures overfitting: an AUC of 0.99 in training and 0.70 in validation says that the model memorizes. It costs a bit more computation time, which is why it is off by default.

</details>

<details><summary>I have 5 folds and I get 5 scores. What is the standard error of my estimate?</summary>

It is not the standard deviation divided by the square root of 5: the folds are not independent (they share training data). The standard deviation across folds is an indicator of stability, not a confidence interval. Repeating the cross-validation with different splits reduces the noise due to the random partition, but the repetitions still use the same data, so it does not turn that spread into a valid confidence interval (corrected estimators exist, such as Nadeau and Bengio's). To compare two models, use the same folds and look at the differences fold by fold.

</details>